# 00 — Set up the test artifacts

This book *tests* an agent it does not build. The System Under Test is the
governed complaint agent from **beyond-prompt-and-pray**, so the chapters
here read two kinds of artifact, neither committed (both git-ignored):

1. **beyond-prompt-and-pray's GMS stores** — built by *that* topic's
   `notebooks/00_setup.ipynb`.
2. **A recorded capstone run** — `data/capstone_{run,rows,testset}.json`,
   produced by `scripts/capstone_run.py`, which drives the agent through the
   DoE framework. The analysis, attribution and resilience chapters read it.

**What you need**

- The licensed **`knowlytix`** substrate (`pip install knowlytix` + a key at
  `~/.knowlytix/license.key`; https://knowlytix.ai/signup/).
- `pip install -e ".[notebooks]"` for matplotlib/pandas used by the plots.
- The capstone campaign runs the agent end to end and **needs an LLM**
  (Qwen + a GPU); the model-free chapters do not.

Stages are **idempotent** — skipped when their output already exists.

## 1. Bootstrap + dependency check

In [ ]:
import importlib.util, os, sys

# Locate this topic's code/ dir, robust to the working directory.
_cwd = os.getcwd()
for REPO in (_cwd, os.path.join(os.path.dirname(_cwd), "code"), os.path.join(_cwd, "code")):
    if os.path.isdir(os.path.join(REPO, "gmstest")):
        break
else:
    REPO = os.environ.get("GMSTEST_REPO", _cwd)
for p in (REPO, os.path.join(REPO, 'scripts')):
    if p not in sys.path:
        sys.path.insert(0, p)
print('repo:', REPO)

if importlib.util.find_spec('knowlytix') is None:
    raise ModuleNotFoundError(
        'knowlytix not found. Install it (`pip install knowlytix`, licensed) '
        'and put your key at ~/.knowlytix/license.key.')

# The SUT lives in the sibling topic; its stores must be built first.
BPP = os.path.normpath(os.path.join(REPO, '..', '..', 'beyond-prompt-and-pray', 'code'))
needed = ['gms_banking_store', 'gms_policy_store', 'gms_regulatory_store']
missing = [n for n in needed if not os.path.isdir(os.path.join(BPP, 'data', n))]
for n in needed:
    print(f"  beyond-prompt-and-pray/{n:24} "
          f"{'OK' if n not in missing else 'MISSING'}")
if missing:
    print('\n-> Run beyond-prompt-and-pray/notebooks/00_setup.ipynb first '
          '(Tier 1 is CPU-only).')

## 1b. Preflight — dependencies and inputs


In [ ]:
# Fail fast, and fail completely: the stages below load models and write stores,
# so a missing dependency should surface here rather than several minutes in.
# Everything wrong is reported at once instead of one error per re-run.
import importlib.util
from pathlib import Path

_problems = []

for _mod, _why in [
    ("knowlytix", "the licensed GMS substrate — every stage below needs it"),
    ("torch", "tensor backend used to build and calibrate the stores"),
]:
    if importlib.util.find_spec(_mod) is None:
        _problems.append(f"missing package {_mod!r} — {_why}")

for _rel in ['data/capstone_run.json', 'catalogs/base_catalog.yaml', 'catalogs/factor_catalog.yaml']:
    if not (Path(REPO) / _rel).exists():
        _problems.append(f"missing input {_rel!r} — expected in a complete clone")

if _problems:
    print("PREFLIGHT FAILED — nothing has been built:\n")
    for _p in _problems:
        print("  -", _p)
    print(
        "\nknowlytix is free to install but gated by a runtime licence key:\n"
        "    pip install knowlytix\n"
        "    # then place your key at ~/.knowlytix/license.key\n"
        "    # sign up at https://knowlytix.ai/signup/\n"
        "Missing inputs usually mean a partial clone or a deleted data/ directory."
    )
    raise RuntimeError("preflight failed; see the list above")

print("preflight ok — dependencies importable, inputs present")


## 2. Idempotent stage runner

In [ ]:
import subprocess, sys, os

def stage(title, script, args=(), produces=()):
    """Run scripts/<script> unless every path in `produces` already exists."""
    produces = list(produces)
    if produces and all(os.path.exists(os.path.join(REPO, p)) for p in produces):
        print(f'\u2713 {title}: already built \u2014 skipping')
        return
    cmd = [sys.executable, os.path.join(REPO, 'scripts', script), *map(str, args)]
    print(f'\u25b6 {title}: python scripts/{script} ' + ' '.join(map(str, args)))
    env = os.environ.copy()
    env['PYTHONPATH'] = REPO + os.pathsep + env.get('PYTHONPATH', '')
    r = subprocess.run(cmd, cwd=REPO, env=env)
    if r.returncode:
        raise RuntimeError(f'{script} failed (exit {r.returncode})')
    print(f'\u2713 {title}: done')

## 3. Capstone campaign (needs the SUT + an LLM)

Drives the governed complaint agent through the DoE framework and records
the run the analysis chapters read. This exercises the real agent, so it
needs beyond-prompt-and-pray's stores (above) and an LLM. Set
`RUN_CAPSTONE = True` to build it.

In [ ]:
RUN_CAPSTONE = False  # set True to record a capstone run (needs the SUT + an LLM)

if RUN_CAPSTONE:
    stage('Capstone campaign', 'capstone_run.py',
          produces=['data/capstone_run.json'])
    print('\nCapstone artifacts ready.')
else:
    print('RUN_CAPSTONE is False. The model-free chapters (taxonomy, design '
          'space, enrichment, resilience) run without it; the analysis and '
          'attribution chapters need data/capstone_run.json.')

## 4. What's present

In [ ]:
import os
for f in ('capstone_run.json', 'capstone_rows.json', 'capstone_testset.json',
          'capstone_companions.json'):
    p = os.path.join(REPO, 'data', f)
    print(f"  {f:28} {'OK' if os.path.isfile(p) else 'missing'}")

## Done

Artifacts live under `code/data/` (git-ignored, never packaged). Re-run this
notebook any time; existing artifacts are skipped.